# 1. LangChain 메시지 처리 방법

> Runnable은 **"무엇을 실행할 것인가"** 를 정의하고, `invoke()` / `stream()` / `batch()`는 **"어떻게 실행할 것인가"** 를 정의합니다.

## 학습 목표

1. Runnable이 무엇인지 설명할 수 있습니다.
2. `invoke` / `stream` / `batch`와 비동기 버전(`ainvoke` / `astream` / `abatch`)의 차이를 이해합니다.
3. Chat 모델에 메시지 리스트를 전달하는 방법을 익힙니다.
4. 동기(Sync)와 비동기(Async) 실행의 차이를 실행 시간으로 확인합니다.

**기준 버전**: LangChain 1.x, `langchain-openai` 1.x, 모델 `gpt-6-luna`

## 1. Runnable

Runnable은 LangChain의 모든 실행 가능한 객체가 따르는 **공통 인터페이스**입니다.

- 입력(Input)을 받습니다.
- 작업을 수행합니다.
- 출력(Output)을 반환합니다.
- 다른 Runnable과 연결할 수 있습니다.

```text
입력 → 처리 → 출력
```

쉽게 말해 Runnable은 **"전기 콘센트 규격"** 과 같습니다. 모든 가전제품이 같은 플러그 규격을 쓰면 서로 연결할 수 있듯이,
LangChain의 Prompt, Model, Output Parser, Retriever, Tool이 모두 Runnable 규격을 따르기 때문에 아래처럼 체인을 쉽게 만들 수 있습니다.

```text
prompt | llm | parser
Prompt → ChatModel → OutputParser
```

### 실습 환경 준비

터미널에서 필요한 패키지를 설치하고, 프로젝트 최상위 폴더의 `.env` 파일에 API 키와 모델명을 저장합니다.

```text
OPENAI_API_KEY="YOUR_OPENAI_API_KEY"
OPENAI_DEFAULT_MODEL="gpt-6-luna"
```

> API 키는 코드 셀이나 GitHub에 직접 입력하지 마세요.

```bash
uv add langchain langchain-openai python-dotenv
```

In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")

# reasoning_effort="none": 수업 예제는 빠른 응답과 낮은 비용을 위해 추론 기능을 끕니다.
llm = ChatOpenAI(model=default_model, reasoning_effort="none")
print("사용 모델:", default_model)

> **gpt-6-luna 사용 시 참고**
> - `reasoning_effort`는 `none` / `low` / `medium`(기본) / `high` / `xhigh` / `max`를 지원합니다. 강의 예제는 `none`으로 통일합니다.
> - `temperature`는 추론이 꺼져 있을 때(`reasoning_effort="none"`)만 적용됩니다.
> - 모델명·요금·지원 파라미터는 바뀔 수 있으므로 수업 전에 [OpenAI 모델 문서](https://developers.openai.com/api/docs/models/gpt-6-luna)를 확인하세요.

## 2. 메시지 처리 방법

`invoke`, `stream`, `batch`는 Runnable을 실행하는 서로 다른 메서드입니다. 실행 구조는 같고, **입력 처리 방식과 출력 반환 방식**이 다릅니다.

```text
동기(Sync)   : 질문 1 실행 → 완료 대기 → 결과 반환 → 질문 2 실행
├─ invoke()
├─ stream()
└─ batch()

비동기(Async): 질문 1 실행 → 기다리지 않음 → 질문 2 실행 → 질문 3 실행 → 결과 수신
├─ ainvoke()
├─ astream()
└─ abatch()
```

FastAPI, Gradio, Streamlit, Agent 시스템을 만들 때는 `ainvoke()`, `astream()`, `abatch()`를 많이 사용합니다.

### 2-1. invoke (1:1)

- Runnable의 가장 기본적인 실행 방법입니다.
- 입력 1개를 받아 **완성된 출력 1개**를 반환합니다.
- 실행이 끝날 때까지 기다린 뒤 최종 결과를 반환합니다.

```text
질문 → 모델 실행 → 완료될 때까지 대기 → 결과 반환
```

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="당신은 서울의 음식과 문화 전문가입니다."),
    HumanMessage(content="서울을 대표하는 음식을 맛볼 수 있는 레스토랑 3곳을 추천해 주세요."),
]

# 튜플 형식으로도 같은 메시지를 만들 수 있습니다.
# messages = [("system", "당신은 서울의 음식과 문화 전문가입니다."),
#             ("human", "서울을 대표하는 음식을 맛볼 수 있는 레스토랑 3곳을 추천해 주세요.")]

response = llm.invoke(messages)
print(response.content)

#### ainvoke() : 비동기 방식

```text
질문 → 모델 실행 → (기다리는 동안 다른 작업 가능) → 결과 수신
```

> Jupyter Notebook에서는 셀에서 바로 `await`를 쓸 수 있습니다. 일반 `.py` 파일에서는 `async def` 함수 안에서 사용하고 `asyncio.run()`으로 실행합니다.

In [ ]:
response = await llm.ainvoke(messages)
print(response.content)

### 2-2. stream (1:다)

- 모델이 생성한 결과를 **청크(chunk) 단위로 나누어** 실시간으로 반환합니다.
- 입력은 1개, 출력은 여러 개의 부분 결과(chunk)입니다.
- 채팅창처럼 답변이 생성되는 과정을 바로 보여줘야 할 때 적합합니다.

```text
Input → Model → Chunk1
                Chunk2
                Chunk3
                ...    → Final
```

In [ ]:
messages = [
    ("system", "당신은 서울의 문화 전문가입니다."),
    ("human", "서울에서 데이트할 때 가 볼 만한 분위기 좋은 곳을 알려 주세요."),
]

for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)

#### astream() : 비동기 방식

- `astream()`은 **비동기 생성기(AsyncGenerator)** 를 반환하므로 반드시 `async for`로 꺼냅니다.
- `async for`는 "아직 도착하지 않은 데이터를 비동기로 기다리면서 하나씩 꺼내는 반복문"입니다.

In [ ]:
async for chunk in llm.astream(messages):
    print(chunk.content, end="", flush=True)

### 2-3. batch (다:다)

- **여러 입력(list)** 에 같은 Runnable을 적용하고, 입력 개수만큼의 결과 리스트를 반환합니다.
- 내부적으로 병렬 처리되어 여러 요청을 효율적으로 처리할 수 있습니다.

```text
입력1 → 실행 → 결과1
입력2 → 실행 → 결과2
입력3 → 실행 → 결과3
```

In [ ]:
batch_messages = [
    [("system", "당신은 서울의 음식 전문가입니다."), ("human", "광장시장에서 먹을 만한 길거리 음식을 소개해 주세요.")],
    [("system", "당신은 서울의 문화 전문가입니다."), ("human", "서울에서 데이트할 때 가 볼 만한 곳을 알려 주세요.")],
]

responses = llm.batch(batch_messages)
for response in responses:
    print(response.content)
    print("-" * 40)

#### abatch() : 비동기 방식

`abatch()`는 결과를 모두 모아 **리스트로 반환**하므로 일반 `for`로 꺼냅니다.

```text
입력1 ─┐
입력2 ─┼─ 동시에 실행 → [결과1, 결과2, 결과3]
입력3 ─┘
```

In [ ]:
responses = await llm.abatch(batch_messages)
for response in responses:
    print(response.content)
    print("-" * 40)

### 2-4. 비동기 키워드 정리

| 키워드 | 의미 |
|---|---|
| `async` | 이 함수에서 비동기 작업을 사용하겠다 |
| `await` | 비동기 작업이 끝날 때까지 기다려라 |
| `for` | 일반 리스트 반복 |
| `async for` | 비동기로 들어오는 데이터를 반복 |

### 2-5. invoke vs stream vs batch 비교

| 구분 | invoke | stream | batch |
|---|---|---|---|
| 입력 개수 | 1개 | 1개 | 여러 개(list) |
| 출력 형태 | 완성된 결과 1개 | 부분 결과(chunk) 여러 개 | 결과 리스트 |
| 반환 시점 | 실행 완료 후 한 번에 | 생성되는 즉시 실시간 | 모든 입력 처리 후 |
| 주요 목적 | 일반 질의응답 | 실시간 UI, 채팅 | 대량 처리, 반복 작업 |
| 사용 예시 | 챗봇 단일 질문 | 스트리밍 채팅창 | 여러 문서 요약 |

| 함수 | 반환값 | 꺼내는 방법 |
|---|---|---|
| `invoke()` | `AIMessage` | 바로 사용 |
| `ainvoke()` | `AIMessage` | `await` |
| `batch()` | `list[AIMessage]` | `for` |
| `abatch()` | `list[AIMessage]` | `await` + `for` |
| `stream()` | Generator | `for` |
| `astream()` | AsyncGenerator | `async for` |

```text
Prompt → LLM → Parser → Tool ...
   ↑ 모두 Runnable ↑
          ↓
 invoke / stream / batch
```

## 3. Chat 모델과 메시지

### 3-1. Completion 모델과 Chat 모델

| 구분 | Completion 모델 (과거 방식) | Chat 모델 (현재 표준) |
|---|---|---|
| 입력 방식 | 하나의 문자열 | 메시지 리스트 |
| 구조 | 단일 프롬프트 | System / Human / AI 역할 구분 |
| 대화 맥락 | 프롬프트에 직접 작성 | 메시지 리스트로 관리 |
| 예시 | `"Explain AI."` | `[SystemMessage(...), HumanMessage(...), AIMessage(...)]` |

> Completion 방식(레거시 Completions API)은 `gpt-6-luna`에서 지원하지 않습니다. 이 강의는 **Chat 모델(`ChatOpenAI`)** 만 사용합니다.

```text
[SystemMessage]
[HumanMessage]
[AIMessage]
     ↓
   Model
     ↓
AIMessage Output
```

### 3-2. 메시지 클래스를 쓰는 이유

- 대화의 **역할(role)** 을 명확히 구분할 수 있습니다.
- 다른 LLM으로 바꿔도 같은 코드 구조를 유지할 수 있습니다.
- 대화 히스토리(멀티턴), 에이전트, 메모리 기반 챗봇에서 맥락을 유지하는 기본 단위입니다.
- 메시지 클래스는 `langchain_core.messages`에서 가져옵니다.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

sys_msg = SystemMessage(content="당신은 음악 추천을 해 주는 전문 AI입니다.")
user_msg = HumanMessage(content="1990년대 재즈 5곡만 추천해 줘.")

ai_msg = llm.invoke([sys_msg, user_msg])
print(ai_msg.content)

이전 대화(`AIMessage`)를 함께 전달하면 모델이 앞의 맥락을 이어서 답합니다.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

conversation = [
    SystemMessage(content="당신은 여행을 추천해 주는 전문 AI입니다."),
    HumanMessage(content="베트남 사파에 1박 2일 여행 가면 충분할까?"),
    AIMessage(content="사파는 하노이에서 이동 시간이 길어 1박 2일은 빠듯합니다. 일정을 늘리거나 다른 여행지를 고려해 보세요."),
    HumanMessage(content="음... 그러면 1박 2일로는 어디가 좋을까?"),
]

ai_msg = llm.invoke(conversation)
print(ai_msg.content)

## 4. 동기(Sync) vs 비동기(Async)

| 구분 | 동기(Sync) | 비동기(Async) |
|---|---|---|
| 실행 방식 | 순차적(Blocking) | 동시(Non-blocking) |
| 효율성 | I/O 작업에 비효율적 | 네트워크·파일 같은 I/O 작업에 효율적 |
| 복잡성 | 단순, 디버깅 쉬움 | 흐름 추적이 상대적으로 어려움 |
| 사용 예시 | 단순 스크립트 | 웹 서버, 여러 API 동시 호출 |
| Python 키워드 | `def`, `return` | `async def`, `await` |

- `async def`로 선언한 함수를 **코루틴(coroutine)** 이라고 합니다.
- `await`는 코루틴 안에서만 쓰며, 기다리는 동안 다른 코루틴이 실행됩니다.
- `asyncio.gather()`는 여러 비동기 작업을 **동시에 실행**하고 모두 끝날 때까지 기다립니다.

| 함수 | 의미 |
|---|---|
| `invoke()` | 요청하고 끝날 때까지 프로그램이 기다림 |
| `ainvoke()` | 비동기 요청(코루틴) 생성 |
| `await` | 비동기 작업이 끝날 때까지 기다림 |
| `asyncio.gather()` | 여러 비동기 작업을 동시에 실행 |

### 4-1. 실행 시간 비교

같은 질문을 3번 보낼 때 동기와 비동기의 실행 시간을 비교합니다. (호출 횟수만큼 비용이 발생하므로 횟수는 작게 유지합니다.)

In [19]:
import asyncio
import time

question1 = [
    ("system", "당신은 서울의 음식 전문가입니다."),
    ("human", "광장시장 대표 음식 한 가지를 한 문장으로 알려 주세요."),
]
question2 = [
    ("system", "당신은 서울의 음식 전문가입니다."),
    ("human", "남대문 시장 대표 음식 한 가지를 한 문장으로 알려 주세요."),
]
question3 = [
    ("system", "당신은 서울의 음식 전문가입니다."),
    ("human", "동대문 시장 대표 음식 한 가지를 한 문장으로 알려 주세요."),
]


In [20]:
##time
# 동기: 하나가 끝나야 다음 호출
result1 = llm.invoke(question1)
result2 = llm.invoke(question2)
result3 = llm.invoke(question3)

In [21]:
##time
# 비동기: N개를 동시에 호출
import asyncio

result1, result2, result3 = await asyncio.gather(
    llm.ainvoke(question1),
    llm.ainvoke(question2),
    llm.ainvoke(question3)
)

**결론**

- 동기(Sync): 작업을 하나씩, 기다리면서 처리합니다. 단순하지만 느릴 수 있습니다.
- 비동기(Async): 응답을 기다리는 동안 다른 요청을 보내므로 여러 호출을 할 때 빠릅니다.
- 실행 시간과 서비스 구조에 맞게 두 방식을 선택합니다.

| 메서드 | 방식 | 입력 수 | 출력 수 | 사용 목적 |
|---|---|---|---|---|
| `invoke()` | 동기 | 1 | 1 | 일반 실행 |
| `ainvoke()` | 비동기 | 1 | 1 | 웹 서비스 |
| `stream()` | 동기 스트리밍 | 1 | 여러 chunk | 실시간 출력 |
| `astream()` | 비동기 스트리밍 | 1 | 여러 chunk | 대규모 실시간 서비스 |
| `batch()` | 동기 배치 | 여러 개 | 여러 개 | 반복 처리 |
| `abatch()` | 비동기 배치 | 여러 개 | 여러 개 | 대량 병렬 처리 |

## 5. 실습 - 해 보기

아래 프롬프트로 LLM을 호출해 보세요.

1. `system`: "당신은 여행 전문가로 고객의 여행 일정에 도움을 줍니다"<br>
   `user`: "부산 여행에서 딱 한 곳만 가 봐야 한다면 어떤 곳인지 알려 주세요"
2. `user`: "부산역에서 그곳에 가는 교통편을 알려 주세요"
3. `user`: "그 근처에서 먹을 만한 음식점들을 추천해 주세요"
4. 위 3개를 하나로 묶어서 실행하기

> 2번, 3번 질문의 "그곳", "그 근처"를 모델이 알아들으려면 무엇이 필요할까요? → **3. Memory**에서 다룹니다.

In [ ]:
# 여기에 실습 코드를 작성하세요.